# 09 - Event Study: Camp Fire 2018

The Camp Fire (November 2018, Butte County CA) was the deadliest and most destructive wildfire in California history. It caused catastrophic air quality across Northern California for weeks. This event study estimates the impact on test scores in affected districts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
# Camp Fire: November 8, 2018; primarily affected Butte County (FIPS 06007)
# and caused extreme smoke across Sacramento Valley and Bay Area
# Smoke was extreme for approximately 2-3 weeks

# Flag districts by smoke exposure in 2018
# Treated: districts with smoke_days_2018 above median
# Control: districts with below-median smoke exposure in 2018

smoke_2018 = panel[panel["year"]==2018][["leaid","smoke_days"]].copy()
if smoke_2018.empty:
    raise RuntimeError("No 2018 data — ensure HMS smoke data covers 2018")

median_smoke_2018 = smoke_2018["smoke_days"].median()
high_smoke = smoke_2018[smoke_2018["smoke_days"] > median_smoke_2018]["leaid"].tolist()

panel_cf = panel.copy()
panel_cf["high_smoke_district"] = panel_cf["leaid"].isin(high_smoke)
panel_cf["post_camp_fire"] = (panel_cf["year"] >= 2018).astype(float)
panel_cf["did_camp_fire"]  = panel_cf["high_smoke_district"].astype(float) * panel_cf["post_camp_fire"]

print(f"High-smoke districts (above median in 2018): {len(high_smoke)}")
print(f"Low-smoke comparison districts: {panel_cf[~panel_cf['high_smoke_district']]['leaid'].nunique()}")

## Event study around 2018

In [ ]:
WINDOW = 4
REF_YEAR = 2017
ALL_YEARS = [y for y in range(REF_YEAR - WINDOW, REF_YEAR + WINDOW + 2) if y != REF_YEAR]

for yr in ALL_YEARS:
    panel_cf[f"dum_{yr}"] = (
        panel_cf["high_smoke_district"] &
        (panel_cf["year"] == yr)
    ).astype(float)

dum_cols = [f"dum_{yr}" for yr in ALL_YEARS]
panel_cf_idx = panel_cf.set_index(["leaid","year"])

es_res = PanelOLS(
    panel_cf_idx["test_score_mean"],
    panel_cf_idx[dum_cols],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

es_df = pd.DataFrame({
    "year":  ALL_YEARS,
    "coef":  es_res.params[dum_cols].values,
    "ci_lo": es_res.conf_int().loc[dum_cols,"lower"].values,
    "ci_hi": es_res.conf_int().loc[dum_cols,"upper"].values,
}).sort_values("year")

fig, ax = plt.subplots(figsize=(10,5))
ax.fill_between(es_df["year"], es_df["ci_lo"], es_df["ci_hi"], alpha=0.2, c="steelblue")
ax.plot(es_df["year"], es_df["coef"], "o-", ms=5, lw=1.8, c="steelblue")
ax.axvline(2018, color="red", ls="--", lw=1.5, label="Camp Fire (Nov 2018)")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Year")
ax.set_ylabel("Coefficient (test score SD)")
ax.set_title("Event Study: Camp Fire 2018 Impact on Test Scores\nHigh vs Low Smoke Exposure Districts")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "09_camp_fire_event_study.png", bbox_inches="tight")
plt.show()